In [1]:
import sys
import math
import mysql.connector
from PyQt5.QtWidgets import (
    QApplication, QWidget, QHBoxLayout, QVBoxLayout, QLabel, QSplitter,
    QFrame, QMenuBar, QAction, QMainWindow, QMessageBox,
    QDialog, QLineEdit, QPushButton, QFileDialog, QStatusBar, QListWidget
)
from PyQt5.QtCore import Qt, QMimeData, QPoint
from PyQt5.QtGui import QDrag, QPainter, QPen, QColor, QPixmap
from PyQt5.QtGui import QDoubleValidator

# Data for the legend (left panel)
elements = [
    ("________________________________", ""),
    ("***     NODES       ***", ""),
    ("________________________________", ""),
    ("dam", "blue triangle.png"),
    ("reservoir", "icon2.jpg"),
    ("pump station", "icon3.png"),
    ("mine", "icon5.png"),
    ("rainwater harvesting", "harvest1.jpg"),
    ("farm block ", "farm block.png"),
    ("stormwater ", "stormwater.png"),
    ("groundwater harvesting", "groundwater harvesting.jpg"),
    ("groundwater source", "red diamond.png"),
    ("ecosystem infrastructure", "ecosystem.jpg"),
    ("mine dam", "brown triangle.png"),
    ("water treatment plant", "blue circle.jpg"),
    ("wastewater treatment", "red circle.png"),
    ("junction", "screenshot14.png"),
    ("subarea", "rectangle1.png"),
    ("water supply area", "pentagon.jpg"),
    ("settlement", "red star.png"),
    ("power station", "power station.png"),
    ("industry", "industry.png"),
    ("agriculture", "agriculture.jpg"),
    ("________________________________", ""),
    ("***       DATA      ***", ""),
    ("________________________________", ""),
    ("water quality monitoring point ", "screenshot1.png"),
    ("streamflow gauging station", "screenshot2.png"),
    ("water meter", "screenshot13.png"),
    ("________________________________", ""),
    ("***       CONNECTORS       ***", ""),
    ("________________________________", ""),
    ("runoff", "runoff1.png"),
    ("raw water abstraction", "raw water1.png"),
    ("treated water", "treated1.png"),
    ("greywater reuse", "grey.png"),
    ("wastewater", "wastee.png"),
    ("streamflow", "streamflow1.png"),
    ("treated wastewater", "treatedwaste1.png"),
    ("feedback loop", "feedback.png"),
]

# Database connection function
def connect_to_db():
    return mysql.connector.connect(
        host="localhost",
        user="root",  # replace with your MySQL username
        password="12345678",  # replace with your MySQL password
        database="project_management"
    )

# Function to save a project
def save_project_to_db(project_name, project_location, labels):
    db = connect_to_db()
    cursor = db.cursor()

    # Insert project
    cursor.execute("INSERT INTO projects (name, location) VALUES (%s, %s)", (project_name, project_location))
    project_id = cursor.lastrowid  # Get the ID of the newly created project

    # Insert elements
    for label in labels:
        cursor.execute(
            "INSERT INTO elements (project_id, text, pos_x, pos_y, capacity, type, latitude, longitude, name) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)",
            (project_id, label['text'], label['pos'][0], label['pos'][1], label.get('capacity'), label.get('type'), 
             label.get('latitude'), label.get('longitude'), label.get('name'))
        )
    db.commit()
    cursor.close()
    db.close()

# Function to load a project
def load_project_from_db(project_id):
    db = connect_to_db()
    cursor = db.cursor(dictionary=True)

    # Load project
    cursor.execute("SELECT * FROM projects WHERE id = %s", (project_id,))
    project = cursor.fetchone()

    # Load elements
    cursor.execute("SELECT * FROM elements WHERE project_id = %s", (project_id,))
    elements = cursor.fetchall()

    cursor.close()
    db.close()

    return project, elements

# Draggable QLabel for legend items
class DraggableLabel(QWidget):
    def __init__(self, text, image_path ):
        super().__init__()
        layout = QHBoxLayout(self)
        layout.setSpacing(0)  # Set spacing between the image and text to 0
        layout.setContentsMargins(0, 0, 0, 0)  # Remove margins around the layout
        # Create QLabel for the image
        self.image_label = QLabel(self)
        if image_path:
            pixmap = QPixmap(image_path).scaled(30, 30, Qt.KeepAspectRatio)  # Adjust size as needed
            self.image_label.setPixmap(pixmap)

        # Create QLabel for the text
        self.text_label = QLabel(text, self)

        # Add image and text labels to layout
        layout.addWidget(self.image_label, alignment=Qt.AlignLeft)  # Align the image to the left
        layout.addWidget(self.text_label, alignment=Qt.AlignLeft)  # Align the text to the left

        self.setLayout(layout)

    def mousePressEvent(self, event):
        if event.button() == Qt.LeftButton:
            drag = QDrag(self)
            mime_data = QMimeData()
            mime_data.setText(self.text_label.text())
            drag.setMimeData(mime_data)
            drag.exec_(Qt.MoveAction)

class MovableLabel(QLabel):
    def __init__(self, text, image_path=None, parent=None):
        super().__init__(parent)
        self.setText(text)
        self.image_path = image_path
        self.setStyleSheet("border: 0.2 px solid black; padding: 0 .5px;")
        self._dragging = False
        self.properties = {}  # Store properties of the element

        if image_path:  # Load image if provided
            self.setPixmap(QPixmap(image_path).scaled(50, 50, Qt.KeepAspectRatio))  # Scale image to fit

    def paintEvent(self, event):
        super().paintEvent(event)  # Call the parent class's paint event
        if self.image_path:
            pixmap = QPixmap(self.image_path).scaled(50, 50, Qt.KeepAspectRatio)
            painter = QPainter(self)
            painter.drawPixmap(0, 0, pixmap)  # Draw the image at the top-left corner

    def mousePressEvent(self, event):
        if event.button() == Qt.LeftButton:
            self._dragging = True
            self._drag_start_position = event.pos()

    def mouseMoveEvent(self, event):
        if self._dragging:
            new_position = self.mapToParent(event.pos() - self._drag_start_position)
            self.move(new_position)

    def mouseReleaseEvent(self, event):
        if event.button() == Qt.LeftButton:
            self._dragging = False

    def mouseDoubleClickEvent(self, event):
        if event.button() == Qt.LeftButton:
            properties_dialog = ElementPropertiesDialog(self, self.parent())
            properties_dialog.populate_fields(self.properties)  # Populate fields with current properties
            if properties_dialog.exec_():
                self.properties = properties_dialog.get_properties()  # Update properties if dialog is accepted
                self.update_tooltip()  # Update tooltip to reflect new properties

    def update_tooltip(self):
        tooltip_text = ", ".join(f"{key}: {value}" for key, value in self.properties.items())
        self.setToolTip(tooltip_text)

    def to_dict(self):
        return {
            'text': self.text(),
            'pos': self.pos().toTuple(),
            'properties': self.properties
        }
    
    @classmethod
    def from_dict(cls, data, parent=None):
        label = cls(data['text'], data.get('image_path'), parent)
        label.setProperties(data['properties'])
        label.move(*data['pos'])
        return label

    def setProperties(self, properties):
        self.properties = properties
        self.update_tooltip()

class Arrow(QFrame):
    def __init__(self, start_point, end_point, color, parent=None):
        super().__init__(parent)
        self.start_point = start_point
        self.end_point = end_point
        self.color = color
        self.setFixedSize(8000, 6000)
        self._dragging_start = False
        self._dragging_end = False
        self.line_style = Qt.SolidLine  # Default line style
        self.curved = False  # Default to not curved

    def paintEvent(self, event):
        painter = QPainter(self)
        pen = QPen(QColor(self.color), 2)
        pen.setStyle(self.line_style)  # Set the line style
        painter.setPen(pen)

        if self.curved:
            # Draw a curved line (you can customize this as needed)
            path = QPainterPath()
            path.moveTo(self.start_point)
            path.cubicTo(self.start_point.x(), self.end_point.y(), self.end_point.x(), self.start_point.y(), self.end_point)
            painter.drawPath(path)
        else:
            painter.drawLine(self.start_point, self.end_point)

        # Draw the arrowhead
        self.draw_arrowhead(painter)

    def draw_arrowhead(self, painter):
        arrow_length = 10
        arrow_angle = 30
        angle = math.atan2(self.end_point.y() - self.start_point.y(), self.end_point.x() - self.start_point.x())
        p1 = QPoint(
            int(self.end_point.x() - arrow_length * math.cos(angle + math.radians(arrow_angle))),
            int(self.end_point.y() - arrow_length * math.sin(angle + math.radians(arrow_angle)))
        )
        p2 = QPoint(
            int(self.end_point.x() - arrow_length * math.cos(angle - math.radians(arrow_angle))),
            int(self.end_point.y() - arrow_length * math.sin(angle - math.radians(arrow_angle)))
        )
        
        painter.drawLine(self.end_point, p1)
        painter.drawLine(self.end_point, p2)

    def setLineStyle(self, style):
        self.line_style = style
        self.update()  # Trigger a repaint to apply the new style

    def setCurved(self, curved):
        self.curved = curved
        self.update()  # Trigger a repaint to apply the new curve setting

    def mousePressEvent(self, event):
        if self.is_near_point(event.pos(), self.start_point):
            self._dragging_start = True
        elif self.is_near_point(event.pos(), self.end_point):
            self._dragging_end = True

    def mouseMoveEvent(self, event):
        if self._dragging_start:
            self.start_point = event.pos()
            self.update()
        elif self._dragging_end:
            self.end_point = event.pos()
            self.update()

    def mouseReleaseEvent(self, event):
        self._dragging_start = False
        self._dragging_end = False

    def is_near_point(self, point1, point2, threshold=10):
        return (point1.x() >= point2.x() - threshold and point1.x() <= point2.x() + threshold and
                point1.y() >= point2.y() - threshold and point1.y() <= point2.y() + threshold)

    def set_start_point(self, point):
        self.start_point = point
        self.update()

    def set_end_point(self, point):
        self.end_point = point
        self.update()

class MainWindow(QMainWindow):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("SWENYA SD-APPLICATION")
        self.setGeometry(100, 100, 1200, 600)

        menubar = self.menuBar()
        file_menu = menubar.addMenu("File")

        new_project_action = QAction("New Project", self)
        new_project_action.triggered.connect(self.new_project)
        file_menu.addAction(new_project_action)

        open_project_action = QAction("Open Project", self)
        open_project_action.triggered.connect(self.open_project)
        file_menu.addAction(open_project_action)

        clear_screen_action = QAction("Clear Screen", self)
        clear_screen_action.triggered.connect(self.clear_workspace)
        file_menu.addAction(clear_screen_action)

        exit_action = QAction("Exit", self)
        exit_action.triggered.connect(self.close)
        file_menu.addAction(exit_action)

        central_widget = QWidget(self)
        layout = QHBoxLayout(central_widget)

        left_panel = self.create_legend_panel()
        self.right_panel = self.create_drop_panel()

        splitter = QSplitter(Qt.Horizontal)
        splitter.addWidget(left_panel)
        splitter.addWidget(self.right_panel)
        splitter.setSizes([100, 1200])

        layout.addWidget(splitter)
        central_widget.setLayout(layout)
        self.setCentralWidget(central_widget)

        self.status_bar = QStatusBar(self)
        self.setStatusBar(self.status_bar)

    def clear_workspace(self):
        # Only clear elements and arrows from the drop area
        self.right_panel.clear()

    def new_project(self):
        dialog = ProjectDialog(self)
        if dialog.exec_():
            project_name, project_location = dialog.get_project_info()
            self.add_current_project(project_name)
            self.save_project(project_name, project_location)
            QMessageBox.information(self, "Success", "Project saved successfully!")

    def open_project(self):
        projects = self.list_projects()
        dialog = ProjectSelectionDialog(projects, self)
        if dialog.exec_():
            project_id = dialog.get_selected_project_id()
            if project_id:
                self.load_project(project_id)


    def save_project(self, project_name, project_location):
        labels = [label.to_dict() for label in self.right_panel.labels]
        save_project_to_db(project_name, project_location, labels)
        QMessageBox.information(self, "Success", "Project saved successfully!")

    def load_project(self, project_id):
        project, elements = load_project_from_db(project_id)
        self.right_panel.clear()  # Clear only the drop area elements, not the arrows

        for element in elements:
            label = MovableLabel.from_dict(element, self.right_panel)
            label.show()
            self.right_panel.labels.append(label)

        self.status_bar.showMessage(f"PROJECT '{project['name']}' IS ACTIVE.")

    def create_legend_panel(self):
        legend_panel = QWidget()
        legend_layout = QVBoxLayout()

        for name in elements:
            if name[1]:  # Check if there is an associated image path
                label = DraggableLabel(name[0], name[1])
            else:
                label = QLabel(name[0])  # For text-only items

            legend_layout.addWidget(label)

        legend_panel.setLayout(legend_layout)
        return legend_panel

    def create_drop_panel(self):
        drop_panel = DropArea(self)
        drop_panel.setStyleSheet("background-color: white;")
        return drop_panel

    def list_projects(self):
        db = connect_to_db()
        cursor = db.cursor(dictionary=True)
        cursor.execute("SELECT * FROM projects")
        projects = cursor.fetchall()
        cursor.close()
        db.close()
        return projects

    def add_current_project(self, project_name):
        if not hasattr(self, 'current_project_menu'):
            self.current_project_menu = self.menuBar().addMenu("Current Project")

        project_action = QAction(project_name, self)
        project_action.triggered.connect(lambda: self.load_project(1))  # Load project with ID 1
        self.current_project_menu.addAction(project_action)

class ProjectDialog(QDialog):
    def __init__(self, parent=None):
        super().__init__(parent)

        self.setWindowTitle("New Project")
        self.setGeometry(100, 100, 400, 200)

        layout = QVBoxLayout()

        self.project_name_input = QLineEdit(self)
        self.project_name_input.setPlaceholderText("Enter Project Name")
        layout.addWidget(self.project_name_input)

        self.project_location_input = QLineEdit(self)
        self.project_location_input.setPlaceholderText("Enter Project Location")
        layout.addWidget(self.project_location_input)

        button_layout = QHBoxLayout()
        save_button = QPushButton("Save", self)
        save_button.clicked.connect(self.accept)
        button_layout.addWidget(save_button)

        layout.addLayout(button_layout)

        self.setLayout(layout)

    def get_project_info(self):
        return self.project_name_input.text(), self.project_location_input.text()

class ProjectSelectionDialog(QDialog):
    def __init__(self, projects, parent=None):
        super().__init__(parent)
        self.setWindowTitle("Select a Project")
        layout = QVBoxLayout()

        self.project_list = QListWidget(self)
        for project in projects:
            self.project_list.addItem(f"{project['id']}: {project['name']}")
        layout.addWidget(self.project_list)

        select_button = QPushButton("Select", self)
        select_button.clicked.connect(self.accept)
        layout.addWidget(select_button)

        self.setLayout(layout)

    def get_selected_project_id(self):
        selected_item = self.project_list.currentItem()
        if selected_item:
            return int(selected_item.text().split(":")[0])
        return None

class DropArea(QFrame):
    def __init__(self, parent=None):
        super().__init__(parent)
        self.setAcceptDrops(True)
        self.setMinimumSize(800, 600)
        self.labels = []
        self.arrows = []

    def dragEnterEvent(self, event):
        if event.mimeData().hasText():
            event.acceptProposedAction()

    def dropEvent(self, event):
        text = event.mimeData().text()
        
        # Check if the dropped text corresponds to a connector
        connector_types = {
            "runoff", "raw water abstraction", "treated water", 
            "greywater reuse", "wastewater", "streamflow", 
            "treated wastewater", "feedback loop"
        }

        if text in connector_types:
            # Define arrow styles based on the connector type
            arrow_styles = {
                "runoff": ("dashed", QColor("brown")),
                "raw water abstraction": ("solid", QColor("brown")),
                "treated water": ("solid", QColor("blue")),
                "greywater reuse": ("dashed", QColor("grey")),
                "wastewater": ("solid", QColor("green")),
                "streamflow": ("solid", QColor("blue")),
                 "treated water": ("solid", QColor("lightgreen")),  # Change to light green
                "feedback loop": ("solid", QColor("red")),  # Change to solid red
            }

            style, color = arrow_styles[text]
            arrow = Arrow(event.pos(), event.pos() + QPoint(50, 0), color.name(), self)
            if style == "dashed":
                arrow.setLineStyle(Qt.DashLine)  # Set the line style to dashed
            elif style == "solid":
                arrow.setLineStyle(Qt.SolidLine)  # Set the line style to solid

            arrow.show()
            self.arrows.append(arrow)
        else:
            # If it's not a connector, create a movable label
            image_path = None
            for element in elements:
                if element[0] == text:
                    image_path = element[1]
                    break
            
            label = MovableLabel(text, image_path, self)
            label.move(event.pos())
            label.show()
            self.labels.append(label)

            # Open properties dialog for the label
            properties_dialog = ElementPropertiesDialog(label, self)
            properties_dialog.exec_()

    def clear(self):
        for label in self.labels:
            label.deleteLater()
        self.labels.clear()
        for arrow in self.arrows:
            arrow.deleteLater()
        self.arrows.clear()
        
class ElementPropertiesDialog(QDialog):
    def __init__(self, label, parent=None):
        super().__init__(parent)
        self.label = label
        self.setWindowTitle("Element Properties")

        layout = QVBoxLayout()

        # Existing fields
        self.capacity_input = QLineEdit(self)
        self.capacity_input.setPlaceholderText("Capacity (m³)")
        self.capacity_input.setValidator(QDoubleValidator(0.0, 1e6, 2))  # Set range and decimals
        layout.addWidget(self.capacity_input)

        self.type_input = QLineEdit(self)
        self.type_input.setPlaceholderText("Description (e.g., earth, concrete)")
        layout.addWidget(self.type_input)

        # New fields for latitude, longitude, and name
        self.latitude_input = QLineEdit(self)
        self.latitude_input.setPlaceholderText("Latitude")
        self.latitude_input.setValidator(QDoubleValidator(-90.0, 90.0, 6))  # Latitude range
        layout.addWidget(self.latitude_input)

        self.longitude_input = QLineEdit(self)
        self.longitude_input.setPlaceholderText("Longitude")
        self.longitude_input.setValidator(QDoubleValidator(-180.0, 180.0, 6))  # Longitude range
        layout.addWidget(self.longitude_input)

        self.name_input = QLineEdit(self)
        self.name_input.setPlaceholderText("Name")
        layout.addWidget(self.name_input)

        button_layout = QHBoxLayout()
        save_button = QPushButton("Save", self)
        save_button.clicked.connect(self.save_properties)
        button_layout.addWidget(save_button)

        cancel_button = QPushButton("Cancel", self)
        cancel_button.clicked.connect(self.reject)
        button_layout.addWidget(cancel_button)

        layout.addLayout(button_layout)

        self.setLayout(layout)

    def populate_fields(self, properties):
        self.capacity_input.setText(properties.get('capacity', ''))
        self.type_input.setText(properties.get('type', ''))
        self.latitude_input.setText(properties.get('latitude', ''))
        self.longitude_input.setText(properties.get('longitude', ''))
        self.name_input.setText(properties.get('name', ''))

    def get_properties(self):
        return {
            'capacity': self.capacity_input.text(),
            'type': self.type_input.text(),
            'latitude': self.latitude_input.text(),
            'longitude': self.longitude_input.text(),
            'name': self.name_input.text()
        }

    def save_properties(self):
        self.label.properties = self.get_properties()  # Save properties to the label
        self.accept()  # Close the dialog
        
# Main application
if __name__ == "__main__":
    app = QApplication(sys.argv)
    main_win = MainWindow()
    main_win.show()
    sys.exit(app.exec_())

SystemExit: 0

C:\Users\Ndivheni\.jupyter\New folder\Lib\site-packages\IPython\core\interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
